# 13장 저장, 추론, 하이퍼파라미터 — 실행을 기록으로 남기기

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `train.py` 와 `infer.py` 를 치고, 기본값·`--lr 0.05 --epochs 3 --run lr005`·`--help` 세 번 실행. `results/` 폴더 열어 보기 | [ch13_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_1_follow.ipynb) |
| 2. 주석 달기 | `train.py` 에서 12장과 달라진 줄에만 주석을 달기(무엇을 위해 추가됐는지) | [ch13_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 12장 `train_mnist.py` 를 열고, 이 장을 보지 않고 세 가지(argparse, 결과 폴더, best/last 저장)를 직접 추가. 그다음 `infer.py` 를 빈 파일에서 작성 | [ch13_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "13"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/13")

12장과 달라진 줄에만 주석을 다세요(무엇을 위해 추가됐는지). 12장 코드는 `code/12/train_mnist.py` 에 있습니다.

## `train.py` — 대상: 파일 전체

In [ ]:
%%writefile train.py
import argparse
import json
import os
import time

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


def parse_args():
    p = argparse.ArgumentParser(description="MNIST MLP 학습")
    p.add_argument("--data-root", default=os.environ.get("JUCOS_DATA", "./data"), help="데이터 폴더")
    p.add_argument("--results", default="results", help="결과를 저장할 상위 폴더")
    p.add_argument("--run", default=None, help="이 실행의 이름 (없으면 시각으로)")
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--hidden", type=int, default=128)
    p.add_argument("--lr", type=float, default=0.1)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--log-every", type=int, default=200)
    return p.parse_args()


class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=128, out_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x.flatten(1))))


def evaluate(model, loader, loss_fn, device):
    """정확도와 평균 손실을 함께 돌려준다."""
    model.eval()
    correct, total_loss = 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += loss_fn(scores, labels).item() * len(labels)
            correct += (scores.argmax(1) == labels).sum().item()
    model.train()
    n = len(loader.dataset)
    return correct / n, total_loss / n


def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    run_name = args.run or time.strftime("%Y%m%d-%H%M%S")
    out_dir = os.path.join(args.results, run_name)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(vars(args), f, ensure_ascii=False, indent=2)

    transform = transforms.ToTensor()
    train_set = datasets.MNIST(args.data_root, train=True, download=True, transform=transform)
    test_set = datasets.MNIST(args.data_root, train=False, download=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=1000)

    model = MLP(hidden=args.hidden).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)

    history = []
    best_acc = 0.0
    for epoch in range(1, args.epochs + 1):
        running, t0 = 0.0, time.time()
        for step, (images, labels) in enumerate(train_loader, start=1):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            running += loss.item()
            if step % args.log_every == 0:
                print(f"  에포크 {epoch} 걸음 {step:4d}  평균 손실 {running / args.log_every:.4f}")
                running = 0.0
        acc, test_loss = evaluate(model, test_loader, loss_fn, device)
        history.append({"epoch": epoch, "test_acc": acc, "test_loss": test_loss, "seconds": time.time() - t0})
        print(f"에포크 {epoch}: 시험 정확도 {acc:.4f}  시험 손실 {test_loss:.4f}")
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), os.path.join(out_dir, "model_best.pt"))
    torch.save(model.state_dict(), os.path.join(out_dir, "model_last.pt"))
    with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump({"best_test_acc": best_acc, "history": history}, f, ensure_ascii=False, indent=2)
    print(f"저장 완료: {out_dir}/  (config.json, model_best.pt, model_last.pt, metrics.json)")


if __name__ == "__main__":
    main()

### 확인

In [ ]:
!python train.py --epochs 1 --run check

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- train.py